In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pkl
#======================================================[Preprocessing]----
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold,cross_val_score
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn import preprocessing
#======================================================[MODELS]-----
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,VotingClassifier,BaggingClassifier
from sklearn.svm import SVC,LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier,AdaBoostClassifier,StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
#====================================================[Evaluation]
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score,precision_score,recall_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
from pathlib import Path

data_path = Path(os.environ.get("HEART_DATA_PATH", "heart.csv")).expanduser()
if not data_path.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {data_path}. Set HEART_DATA_PATH to the CSV file path."
    )

df = pd.read_csv(data_path)


In [ ]:
df.head()

Age: age of the patient [years]
Sex: sex of the patient [M: Male, F: Female]
hestPainType: chest pain type [TA: TypicalAngina , ATA: Atypical Angina, NAP: Non-Anginal Pain, ASY: Asymptomatic]

RestingBP: resting blood pressure [mm Hg]

Cholesterol: serum cholesterol [mm/dl]good 200-240

FastingBS: to diagnose prediabetes, diabetes [1: if FastingBS > 120 mg/dl, 0: otherwise]

RestingECG: resting electrocardiogram results اختبار غير جراحي يمكنه الكشف عن التشوهات بما في ذلك عدم انتظام ضربات القلب، والأدلة على مرض القلب التاجي، وتضخم البطين الأيسر، وكتل الفروع الحزيمية

MaxHR: maximum heart rate achieved [Numeric value between 60 and 202]60-100

ExerciseAngina: pain in the chest that comes on with exercise [Y: Yes, N: No]

Oldpeak:ST[Numeric value measured in depression]indicate health conditions such as hypokalemiaنقص بوتاسيوم, myocardial ischemiaنقص ترويه الدم

ST_Slope: the slope of the peak exercise ST segment [Up: upsloping, Flat: flat, Down: downsloping]

HeartDisease: output class [1: heart disease, 0: Normal]

In [ ]:
df.info()

In [ ]:
df['FastingBS'].unique()

In [ ]:
def info_col(col):
    print(f"Column Name: {col.name}")
    print(f"Data Type: {col.dtype}")
    print(f"Number of Unique Values: {col.nunique()}")
    print(f"Null Values: {col.isnull().sum()}")

    if col.nunique() < 10:
       print(f"Unique Values: {col.unique()}")
       print(f"Value Counts:\n{col.value_counts()}\n")
    print("="*50,'\n')


In [ ]:
for col in df.columns:
    info_col(df[col])

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().T

In [ ]:
df.describe(include = 'O').T

In [ ]:
zero_val=df.Cholesterol==0
zero_val.sum()

In [ ]:
num_coulmn=['Cholesterol','RestingBP','MaxHR','Age','Oldpeak']
plt.figure(figsize=(15,10))
for i,col in enumerate(num_coulmn):
     plt.subplot(2,3,i+1)
     hist=sns.histplot(data=df, x=col, hue='HeartDisease', kde=True,bins=20,alpha=0.5,palette=['green','red'])
     for patch in hist.patches:

      x = patch.get_x() + patch.get_width() / 2
      y = patch.get_height()

      hist.text(x, y, f'{int(y)}', ha='center', va='bottom', fontsize=10, color='black')
plt.show()

In [ ]:
#your heart rate can increase from 50% to 90% of your maximum heart rate

In [ ]:
hist=sns.histplot(data=df, x='MaxHR', hue='ExerciseAngina', kde=True,bins=30,alpha=0.5,palette=['green','red'])
for patch in hist.patches:

    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()

    hist.text(x, y, f'{int(y)}', ha='center', va='bottom', fontsize=10, color='black')

In [ ]:
# to see the relation between the [HeartRate] and [ExerciseAngina]

In [ ]:
df['FastingBS']=df['FastingBS'].astype('object')
cat_features=df.select_dtypes(include='object').columns
cat_features

In [ ]:
plt.figure(figsize=(12,12))
for i,col in enumerate(cat_features):
    plt.subplot(3,2,i+1)
    sns.countplot(data = df ,x = col, hue = 'HeartDisease',palette=['green','red'],alpha=.7)

    for p in plt.gca().patches:
        plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

    plt.xlabel(col)
    plt.ylabel('Count')
    plt.title(f'Heart Disease by {col}')
    plt.legend(['No Disease','Disease'])


plt.tight_layout(w_pad=2,h_pad=2)
plt.show()

In [ ]:
df['FastingBS']=df['FastingBS'].astype('int')

In [ ]:
cat_features=df.select_dtypes(include='object').columns
cat_features

### **textEncoding**

In [ ]:
for col in cat_features:
  if df[col].nunique() == 2:
    if col=='Sex':
      df[col]=df[col].map({'M':1,'F':0})
    else:
      df[col]=df[col].map({'N':0,'Y':1})
  else:
    le=LabelEncoder()
    df[col]=le.fit_transform(df[col])
    Path('artifacts').mkdir(exist_ok=True)
    with open(Path('artifacts') / f'{col}_le.pkl', 'wb') as f:
      pkl.dump(le, f)


df.head()

In [ ]:
sns.histplot(df['Age'],kde=True)
plt.show()

In [ ]:
df['Age']=pd.qcut(df['Age'], 10, labels=False)

In [ ]:
df['Age'].value_counts().sort_index()

In [ ]:
correlation = df.corr()
plt.figure(figsize=(10,7))
sns.heatmap(correlation, annot= True, cmap = 'RdBu',linecolor='k',linewidths=.5, mask=correlation.abs()<0.2)

In [ ]:
df.Sex.value_counts()

In [ ]:
pd.DataFrame(df.groupby(['Sex','HeartDisease'])['HeartDisease'].count())

In [ ]:
sns.countplot(x='ExerciseAngina', data=df, palette=['blue','red'],hue='Sex',alpha=.8)

for p in plt.gca().patches:
    plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

plt.xlabel('datection')
plt.ylabel('Count')
plt.title('Heart proplems by gender')
plt.legend(['Female','Male'])
plt.show()

### Check Unbalance

In [ ]:
df.HeartDisease.value_counts()

## Treat Unbalance

In [ ]:
from collections import Counter
x=df.drop('HeartDisease',axis=1)
y=df['HeartDisease']

In [ ]:
#from imblearn.under_sampling import TomekLinks
#random=TomekLinks()
#x_res,y_res=random.fit_resample(x,y)


### splitting

In [ ]:
# So you dont have to tomeklink because tha data is already small

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x , y, test_size=0.30, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train,y_train, test_size=0.25, random_state=42)

In [ ]:
x_train.shape, x_val.shape, x_test.shape

In [ ]:
Accuracy_models=[]
recall_models=[]
Models_names=[]
f1_score_models=[]

#Evaluate Model & check overfitting

In [ ]:
def check_overfitting(model,x_train,x_val,x_test,y_train,y_val,y_test):
    y_train_pred=model.predict(x_train)
    y_val_pred=model.predict(x_val)
    y_test_pred=model.predict(x_test)

    Train_accuracy = accuracy_score(y_train,y_train_pred)
    Validaion_accuracy = accuracy_score(y_val,y_val_pred)
    Test_accuracy = accuracy_score(y_test,y_test_pred)

    test_recall_score=recall_score(y_test,y_test_pred)
    test_f1_score =f1_score(y_test,y_test_pred)
    precision_score_test=precision_score(y_test,y_test_pred)

    print(f"-Train accuracy: {Train_accuracy*100:.2f}%")
    print(f"-Validation accuracy: {Validaion_accuracy*100:.2f}%")
    print(f"-Test accuracy: {Test_accuracy*100:.2f}%\n")

    print(f'-Train Recall: {recall_score(y_train,y_train_pred)*100:.2f}%')
    print(f'-Validation Recall: {recall_score(y_val,y_val_pred)*100:.2f}%')
    print(f'-Test Recall: {test_recall_score*100:.2f}%\n')

    print(f'-Train F1_Score: {f1_score(y_train,y_train_pred)*100:.2f}%')
    print(f'-Validation F1_Score: {f1_score(y_val,y_val_pred)*100:.2f}%')
    print(f'-Test F1_Score: {test_f1_score*100:.2f}%\n')


    print(f'-Accuracy: {Test_accuracy*100:.2f}%')
    print(f'-Recall: { test_recall_score*100:.2f}%')
    print(f'-F1_Score: {test_f1_score*100:.2f}%')
    print(f'-Precision: {precision_score_test*100:.2f}%\n\n')

    sns.heatmap(confusion_matrix(y_test,y_test_pred),annot=True,fmt='d')
    plt.show()

    if abs(Validaion_accuracy - Train_accuracy)*100 > 5:
        print(f"-Overfitting with diff: {abs(Validaion_accuracy - Train_accuracy)*100:.2f}%\n")
        print('xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx\n')
    else:
        print(f"-Not Overfitting & Recall-score: {test_recall_score*100:.2f}%\n\n")

    recall_models.append(test_recall_score)
    Accuracy_models.append(Test_accuracy)
    f1_score_models.append(test_f1_score)

#[1]- Decision Tree

In [ ]:
decision_tree=DecisionTreeClassifier(random_state=42)
params={'criterion':['gini','entropy'], 'max_depth':[4,5,8,10],'ccp_alpha':[.01,.02,.03]}
grid=GridSearchCV(estimator=decision_tree,
                  param_grid=params,
                  cv=5,
                  scoring='recall')
grid.fit(x_train,y_train)

In [ ]:
final_decision_tree=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(final_decision_tree,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('Decision Tree')

**Scaling**

In [ ]:
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_val=scaler.transform(x_val)
x_test=scaler.transform(x_test)

Path('artifacts').mkdir(exist_ok=True)
with open(Path('artifacts') / 'scaler.pkl', 'wb') as f:
    pkl.dump(scaler, f)

# Preferably after Decision Tree
# because scaling decrease the performance of Decision Tree

#[2]-Logistic Regression

In [ ]:
lr=LogisticRegression(solver='liblinear')
params={'C':[0.1,1,10,100],'penalty':['l1','l2']}
grid=GridSearchCV(estimator=lr,
                  param_grid=params,
                  cv=5,
                  scoring='recall')
grid.fit(x_train,y_train)

In [ ]:
finall_lr=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(finall_lr,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('Logistic Regression')

#[3]-SVC

In [ ]:
svc = SVC()

skf = StratifiedKFold(n_splits=5)

recall_scores = cross_val_score(estimator=svc,
                                X=x_train,
                                y=y_train,
                                cv=skf,
                                scoring='recall')  # Use recall as the scoring metric

recall_scores

In [ ]:
svc.fit(x_train,y_train)

In [ ]:
check_overfitting(svc,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('SVC')

#[4]-KNN

In [ ]:
knn=KNeighborsClassifier()

params={'n_neighbors':[1,3,5,7,8,9,11,13,15,17],
        'weights':['uniform','distance'],
        'algorithm':['auto','ball_tree','kd_tree','brute'],
        'p':[1,2],
        }
grid=GridSearchCV(estimator=knn,
                  param_grid=params,
                  cv=5,
                  scoring='recall')
grid.fit(x_train,y_train)

grid.best_params_

In [ ]:
KNN_Modle=grid.best_estimator_

In [ ]:
check_overfitting(KNN_Modle,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('KNN')

#[5]-Gaussian NB

In [ ]:
GNB=GaussianNB()
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}
grid=GridSearchCV(estimator=GNB,
                  param_grid=param_grid,
                  cv=5,
                  scoring='recall')
grid.fit(x_train,y_train)

In [ ]:
Nave_Modle=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(Nave_Modle,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('Nave_Modle')

#[6]-Random forest

In [ ]:
RF=RandomForestClassifier(random_state=42)

param= {
    'n_estimators': [10,50,100,150,200],           # Number of trees
    'max_depth': [1,2,3,4,5,6,7,8,9,10,20,30],           # Maximum depth of trees
    'min_samples_split': [2, 5, 10],           # Minimum samples to split a node
    'min_samples_leaf': [1, 2, 4],             # Minimum samples to be at a leaf node

}
grid=GridSearchCV(estimator=RF,
                  param_grid=param,
                  cv=5,
                  scoring='recall',
                  )

grid.fit(x_train,y_train)

In [ ]:
Random_F_Model=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(Random_F_Model,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('Random forest')

#[7]-Gradient_boosting

In [ ]:
gradient_boosting=GradientBoostingClassifier(n_estimators=200,
                                            learning_rate=0.001,
                                            max_depth=4,
                                            random_state=42)
gradient_boosting.fit(x_train,y_train)

In [ ]:
check_overfitting(gradient_boosting,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('Gradient_boosting')

#[8]-xgboost

In [ ]:
xgboost=XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.01, random_state=42)
xgboost.fit(x_train,y_train)

In [ ]:
check_overfitting(xgboost,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('xgboost')

#[9]-AdaBoost

In [ ]:
adaboost=AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),
                            n_estimators=100,
                            learning_rate=0.01,
                            random_state=42)
adaboost.fit(x_train,y_train)

In [ ]:
check_overfitting(adaboost,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('AdaBoost')

#[10]-Stacking

In [ ]:
from re import X
base_models = [('rf',RandomForestClassifier(max_depth= 4, n_estimators= 40)),
               ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, random_state=42))]

meta_model = XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.01, random_state=42)
stacking = StackingClassifier(estimators=base_models,
                              final_estimator=meta_model
                              )
stacking.fit(x_train, y_train)

In [ ]:
check_overfitting(stacking,x_train,x_val,x_test,y_train,y_val,y_test)

In [ ]:
Models_names.append('Stacking')

#**Accuracy Compare**

In [ ]:
model_accuracy=pd.DataFrame({'Model':Models_names[:10],'Accuracy':Accuracy_models[:10],'Recall':recall_models[:10],'F1_Score':f1_score_models[:10]}).sort_values(by='Recall',ascending=False)
model_accuracy

In [ ]:
plt.figure(figsize=(12,5))
#sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['Accuracy']*100,marker='o',color='b',lw=2,label='Accuracy')
#sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['Recall']*100,marker='o',color='r',lw=2,label='Recall')
#sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['F1_Score']*100,marker='o',color='g',lw=2,label='F1-Score')
sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['Accuracy']*100,marker='o',color='b',lw=2,label='Accuracy')
for i, score in enumerate(model_accuracy['Accuracy']):
    plt.text(i, score*100 + 0.2, f'{score*100:.1f}', ha='center', color='b')

sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['Recall']*100,marker='o',color='r',lw=2,label='Recall')
for i, score in enumerate(model_accuracy['Recall']):
    plt.text(i, score*100 + 0.2, f'{score*100:.1f}', ha='center', color='r')

sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['F1_Score']*100,marker='o',color='g',lw=2,label='F1-Score')
for i, score in enumerate(model_accuracy['F1_Score']):
    plt.text(i, score*100 + 0.2, f'{score*100:.1f}', ha='center', color='g')
#---------------------------------------------------------------

plt.legend()
plt.xticks(rotation=45)

plt.xlabel('Models Names')
plt.ylabel('Accuracy')
plt.title('Accuracy of models')
plt.show()

In [ ]:
best_accuracy=model_accuracy.sort_values(by='Accuracy',ascending=False).reset_index(drop=True)
best_recall=model_accuracy.sort_values(by='Recall',ascending=False).reset_index(drop=True)
best_f1_score=model_accuracy.sort_values(by='F1_Score',ascending=False).reset_index(drop=True)

In [ ]:
print(f'Best Accuracy Model: {best_accuracy.Model[0]} With accuracy: {best_accuracy.Accuracy[0]*100:0.2f}%')
print(f'Best Recall Model: {best_recall.Model[0]} With Recall: {best_recall.Recall[0]*100:0.2f}%')
print(f'Best F1_Score Model: {best_f1_score.Model[0]} With F1_Score: {best_f1_score.F1_Score[0]*100:0.2f}%')

In [ ]:
models=[final_decision_tree, finall_lr, svc, KNN_Modle, Nave_Modle, Random_F_Model, gradient_boosting, xgboost, adaboost, stacking]
names=['Decision_Tree', 'Logistic_Regression','SVC', 'KNN', 'Naive_bayes', 'Random_forest', 'Gradient_boosting', 'xgboost', 'AdaBoost', 'Stacking']
Path('artifacts').mkdir(exist_ok=True)
for model,name in zip(models,names):
    with open(Path('artifacts') / f'{name}.pkl', 'wb') as f:
        pkl.dump(model, f)

In [ ]:
Models_names